# Simulating Orbits Around a Black Hole

## 1. Newtonian Gravity

In Newtonian physics, gravity is a **force pulling objects toward the center**:

$$
\vec F = - \frac{G M m}{r^2} \hat r
$$

The corresponding acceleration is:

$$
\vec a = - \frac{G M}{r^2} \hat r
$$

- $M$ = mass of the central object (e.g., black hole)  
- $r$ = distance from the center  
- $\hat r$ = unit vector pointing from the object toward the center  

The solution for a bound orbit is an **ellipse**:

$$
r(\theta) = \frac{a(1-e^2)}{1 + e \cos\theta}
$$

- $a$ = semi-major axis  
- $e$ = eccentricity ($0 =$ circle, $0 < e < 1 =$ ellipse)  
- $\theta$ = angle around the center  

**Properties:**

- Orbits are perfectly closed (repeat exactly each cycle)  
- Particle moves faster near periapsis and slower near apoapsis

---

## 2. General Relativity (GR)

In GR, **gravity is curvature of spacetime**, not a force.  

Particles follow the **geodesic equation**:

$$
\frac{d^2 x^\mu}{d\tau^2} + \Gamma^\mu_{\alpha \beta} \frac{dx^\alpha}{d\tau} \frac{dx^\beta}{d\tau} = 0
$$

- $x^\mu = (t, r, \phi)$ = coordinates  
- $\tau$ = proper time along the path  
- $\Gamma^\mu_{\alpha\beta}$ = Christoffel symbols (describe how spacetime curves)

### 2a. Understanding Einstein Summation Notation

The repeated indices in

$$
\Gamma^\mu_{\alpha \beta} \frac{dx^\alpha}{d\tau} \frac{dx^\beta}{d\tau}
$$

mean we **sum over the repeated indices**:

$$
\Gamma^\mu_{\alpha \beta} \frac{dx^\alpha}{d\tau} \frac{dx^\beta}{d\tau} 
= \sum_{\alpha=0}^{3} \sum_{\beta=0}^{3} \Gamma^\mu_{\alpha \beta} \frac{dx^\alpha}{d\tau} \frac{dx^\beta}{d\tau}
$$

- $\mu$ = the coordinate we are updating (e.g., $r$ or $\phi$)  
- $\alpha, \beta$ = summed over all coordinates $t, r, \theta, \phi$  
- This is just **a fancy way to write a double sum** compactly.

---

### 2b. Christoffel Symbols for Planar Schwarzschild Orbits

For motion restricted to a plane ($\theta = \pi/2$):

$$
\begin{aligned}
\Gamma^r_{tt} &= \frac{M(r-2M)}{r^3}, &
\Gamma^r_{rr} &= - \frac{M}{r(r-2M)}, &
\Gamma^r_{\phi\phi} &= -(r-2M) \\
\Gamma^\phi_{r\phi} &= \Gamma^\phi_{\phi r} = \frac{1}{r}, &
\Gamma^t_{tr} &= \Gamma^t_{rt} = \frac{M}{r(r-2M)}
\end{aligned}
$$

The GR equations of motion in polar coordinates are:

$$
\begin{aligned}
\frac{d v_r}{d\tau} &= -\Gamma^r_{tt} v_t^2 - \Gamma^r_{rr} v_r^2 - \Gamma^r_{\phi\phi} v_\phi^2 \\
\frac{d v_\phi}{d\tau} &= -2 \Gamma^\phi_{r\phi} v_r v_\phi \\
\frac{d v_t}{d\tau} &= -2 \Gamma^t_{tr} v_t v_r
\end{aligned}
$$

- $v_r$ = radial velocity  
- $v_\phi$ = angular velocity  
- $v_t$ = rate of change of coordinate time  



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Utility functions

def ellipse_initial_conditions(a, e, M):
    """
    Return initial x, y, vx, vy for a Newtonian ellipse starting at periapsis
    """
    r = a * (1 - e)
    x = r
    y = 0
    v = np.sqrt(M * (2/r - 1/a))  # vis-viva
    vx = 0
    vy = v
    return x, y, vx, vy

def cartesian_to_polar(x, y):
    r = np.sqrt(x**2 + y**2)
    phi = np.arctan2(y, x)
    return r, phi

def polar_velocity(x, y, vx, vy):
    """
    Convert Cartesian velocities to polar (vr, vphi)
    """
    r = np.sqrt(x**2 + y**2)
    vr = (x*vx + y*vy)/r
    vphi = (x*vy - y*vx)/r**2
    return vr, vphi

def polar_to_cartesian(r, phi):
    x = r * np.cos(phi)
    y = r * np.sin(phi)
    return x, y

# Parameters
M = 1      # central mass
a = 20     # semi-major axis
e = 0.4    # eccentricity
dt = 0.001
steps = 50000

# Initial conditions
x, y, vx, vy = ellipse_initial_conditions(a, e, M)

# Newtonian orbit
x_newton, y_newton = [], []

x_n, y_n, vx_n, vy_n = x, y, vx, vy

for _ in range(steps):
    r = np.sqrt(x_n**2 + y_n**2)
    ax = -M * x_n / r**3
    ay = -M * y_n / r**3
    vx_n += ax*dt
    vy_n += ay*dt
    x_n += vx_n*dt
    y_n += vy_n*dt
    x_newton.append(x_n)
    y_newton.append(y_n)

# General Relativity orbit

# Start at the same position
x_gr, y_gr = x, y
vr, vphi = polar_velocity(x_gr, y_gr, vx, vy)
r, phi = cartesian_to_polar(x_gr, y_gr)
vt = 1  # dt/dτ, can be kept 1 for this simple demo

x_gr_list = []
y_gr_list = []

for _ in range(steps):
    # Christoffel symbols for Schwarzschild (planar motion)
    Gamma_r_tt = M*(r-2*M)/r**3
    Gamma_r_rr = -M/(r*(r-2*M))
    Gamma_r_pp = -(r-2*M)
    Gamma_t_tr = M/(r*(r-2*M))
    Gamma_p_rp = 1/r

    # accelerations
    ar = -Gamma_r_tt*vt**2 - Gamma_r_rr*vr**2 - Gamma_r_pp*vphi**2
    aphi = -2*Gamma_p_rp*vr*vphi
    at = -2*Gamma_t_tr*vt*vr

    # update velocities
    vr += ar*dt
    vphi += aphi*dt
    vt += at*dt

    # update position
    r += vr*dt
    phi += vphi*dt
    t = 0  # optional, not used in plotting

    x_gr, y_gr = polar_to_cartesian(r, phi)
    x_gr_list.append(x_gr)
    y_gr_list.append(y_gr)

# =================================================

# Plotting
plt.figure(figsize=(6,6))

# Newtonian orbit
plt.plot(x_newton, y_newton, label="Newtonian", color="blue")

# GR orbit
plt.plot(x_gr_list, y_gr_list, label="General Relativity", color="red")

# Black hole
plt.scatter(0, 0, s=200, color='black', label="Black Hole")

# Event horizon
theta = np.linspace(0,2*np.pi,200)
plt.plot(2*M*np.cos(theta), 2*M*np.sin(theta), 'k--', label="Event Horizon")

plt.gca().set_aspect('equal', adjustable='box')
plt.xlim(-25,25)
plt.ylim(-25,25)
plt.legend()
plt.show()